<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Daily_Challenge_LangChain_OpenSource_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: LangChain Pipelines with Open-Source LLMs — Completed

This notebook completes the guided challenge using a lightweight open-source model that can run on CPU (`google/flan-t5-small`).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [ ]:
# Optional hardware check
# This cell works in Colab/Kaggle. It prints GPU info if available, otherwise confirms CPU runtime.

import platform

print("Python:", platform.python_version())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("CPU runtime")
except Exception as error:
    print("CPU runtime or PyTorch not installed yet:", error)


In [ ]:
# Install dependencies
# Pinned versions keep LLMChain and ConversationChain behavior stable for this course notebook.
%pip install -q "transformers==4.37.2" "langchain==0.1.7" "langchain-community==0.0.20" "langchain-core==0.1.23" sentencepiece accelerate


## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [ ]:
# Import libraries
import time
import warnings

warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain


In [ ]:
# Choose a small public model
# google/flan-t5-small is light, CPU-friendly, and works well for instruction-style prompts.
model_name = "google/flan-t5-small"
print("Selected model:", model_name)


In [ ]:
# Load tokenizer and model
# This may take a short time the first time because the model is downloaded from Hugging Face.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer and model loaded successfully.")


In [ ]:
# Create a Hugging Face generation pipeline and wrap it for LangChain
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
)

llm = HuggingFacePipeline(pipeline=gen_pipeline)
print("LangChain HuggingFacePipeline wrapper is ready.")


In [ ]:
# Build a PromptTemplate + LLMChain for friendly rewriting
rewrite_template = """You are a helpful tutor.
Rewrite the text below to make it simpler for beginners.
Keep the meaning, use plain language, and keep it short.

Text: {text}

Simplified rewrite:"""

rewrite_prompt = PromptTemplate(
    template=rewrite_template,
    input_variables=["text"],
)

rewrite_chain = LLMChain(prompt=rewrite_prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."

start = time.time()
rewritten = rewrite_chain.run(text=sample_text)
elapsed = time.time() - start

print("Original text:")
print(sample_text)
print("\nRewritten text:")
print(rewritten)
print(f"\nLatency: {elapsed:.2f} seconds")


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [ ]:
from langchain.schema.runnable import RunnableLambda
from langchain.prompts import PromptTemplate

# Prompt 1: summarize a paragraph
summary_prompt = PromptTemplate(
    template="""Summarize the paragraph below in one clear and short sentence.

Paragraph:
{paragraph}

Summary:""",
    input_variables=["paragraph"],
)

# Prompt 2: transform the summary into exactly 3 bullet points
bullets_prompt = PromptTemplate(
    template="""Turn the summary below into exactly 3 short bullet points.
Use simple language.

Summary:
{summary}

3 bullet points:""",
    input_variables=["summary"],
)

print("Summary prompt and bullet prompt are ready.")


In [ ]:
# First stage: paragraph -> summary string
summary_chain = summary_prompt | llm

# Full LCEL pipeline:
# 1. Receives {"paragraph": "..."}
# 2. Runs summary_chain to create {"summary": "..."}
# 3. Sends that summary into bullets_prompt
# 4. Sends the bullet prompt into the same LLM
summarize_then_bullets = (
    {"summary": summary_chain}
    | bullets_prompt
    | llm
)

print("Two-step pipeline is ready.")


In [ ]:
paragraph = """LangChain is a framework for building applications with large language models.
It helps developers connect prompts, models, tools, memory, and external data sources.
It supports chains, agents, retrieval workflows, and runnable pipelines, making it easier
to build practical AI applications."""

start = time.time()
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
elapsed = time.time() - start

print("Input paragraph:")
print(paragraph)
print("\nPipeline output:")
print(bullets_output)
print(f"\nLatency: {elapsed:.2f} seconds")


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [ ]:
# Bonus: build a simple conversation chain with memory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

conversation_template = """The following is a conversation between a student and an AI assistant.
The assistant is concise, encouraging, and beginner-friendly.

Current conversation:
{history}

Human: {input}
AI:"""

conversation_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=conversation_template,
)

memory = ConversationBufferMemory()

convo = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=conversation_prompt,
    verbose=False,
)

reply1 = convo.predict(input="Hi there! I am learning LangChain. What is it?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")

print("Turn 1:")
print(reply1)
print("\nTurn 2:")
print(reply2)
print("\nMemory buffer:")
print(memory.buffer)


## Observations

- **Latency:** `google/flan-t5-small` is light enough for CPU. On Colab/Kaggle CPU, each response may take a few seconds; on GPU it is usually faster.
- **Quality:** The model is good for simple rewriting, summarizing, and short bullet points, but it is not as powerful as larger instruction models.
- **Quirks:** Because the model is tiny, it can sometimes produce very short, repetitive, or incomplete answers. Clear prompt instructions improve the output.
- **Memory behavior:** In the bonus conversation, `ConversationBufferMemory` stores the first user turn and makes it available to the second turn, so the model can answer the follow-up with context.
